# Rare Variant Annotation and Structural Analysis Pipeline

**Author:** Jordan Manning  
**Degree:** MSc Health Data Science, University of St Andrews  
**Date:** August 2026

## Overview
The prediction resources are used as follows:

- **PrimateAI** — annotations are obtained by matching variants against a locally downloaded PrimateAI score database.
- **AlphaMissense** — annotations are obtained by matching variants against a locally downloaded, compressed AlphaMissense score database.
- **EVE** — annotations are obtained by searching locally downloaded, gene-specific EVE CSV files.
- **UniProt REST API** — used in the structural workflow to identify reviewed human UniProt for target genes.
- **Ensembl REST API / Variant Effect Predictor (VEP)** — used to translate genomic variant coordinates into amino-acid positions.
- **AlphaFold Protein Structure Database** — the structural workflow downloads the corresponding AlphaFold PDB structure after the UniProt has been identified.
- **PyMOL** — the `.pml` scripts provide reproducible instructions for visualising and highlighting mapped variant residues.

The pipelines are applied to two variant datasets:

1. **WES Study (DYS)** 
2. **Twin Study (TWIN)** 

A separate targeted variant table is used for the three-dimensional structural workflow.

> **Important:** PrimateAI, AlphaMissense and EVE are **not queried through APIs in this notebook**. Their downloadable databases must be available locally before those sections can be run. The structural section is the part of the workflow that communicates with external REST services.


## Before running

The notebook expects the following user-specific input files to be available in the working directory:

- `ExtractedDYS.xlsx` — WES/dyslexia variants.
- `ExtractedTWIN.xlsx` — twin-study variants.
- `Targeted_Variants.csv` — variants selected for 3D structural analysis.

The following **external prediction resources are separate local downloads** and are required by the corresponding sections:

- `PrimateAI_scores_v0.2_hg38.tsv`
    - https://basespace.illumina.com/projects/51955905/about 
- `AlphaMissense_hg38.tsv.gz`
    - https://console.cloud.google.com/storage/browser/dm_alphamissense;tab=objects?prefix=&forceOnObjectsSortingFiltering=false 
- The EVE bulk dataset
    - https://evemodel.org/download/bulk 

The 3D structural workflow does not require a pre-downloaded UniProt or Ensembl database. It queries the UniProt and Ensembl REST services at runtime and downloads the required AlphaFold PDB files into the `pdb_files/` directory.

### Expected Python packages

The notebook uses:

- `pandas`
- `requests`
- `re` (Python standard library)
- `glob` (Python standard library)
- `os` (Python standard library)
- `time` (Python standard library)

The notebook does not install packages automatically.

> **Path configuration:** The EVE directory is currently constructed from the user's home directory. On another computer, modify the path components in the EVE section so that they point to the extracted EVE `variant_files` directory.

# 1. WES Study (DYS)

The first section annotates the WES study variants. The Excel files are read into pandas dataframes and matched against external prediction databases using either genomic coordinates and alleles, or protein-level variant descriptions depending on the annotation tool.

## 1.1 PrimateAI

### Method

The PrimateAI pipeline:

1. Reads the WES variant table from Excel.
2. Reads the PrimateAI tab-separated database, skipping its metadata rows.
3. Matches chromosome and genomic-position formats.
4. Matches variants using chromosome, position, reference allele and alternate allele.
5. Retains the original WES columns and adds the PrimateAI score.
6. Saves the annotated dataset as a CSV file.

The database is read with `on_bad_lines='skip'`, matching the behaviour of the original script.

In [ ]:
import pandas as pd


def perform_primate_annotation(input_excel, primate_db_tsv, output_csv):
    """Annotate WES variants with PrimateAI scores.
    input_excel : str
        Path to the Excel file containing the WES variants.
    primate_db_tsv : str
        Path to the PrimateAI tab-separated database.
    output_csv : str
        Path for the annotated CSV output.
    """

    # ------------------------------------------------------------------
    # 1. Load the WES variant data
    # ------------------------------------------------------------------
    print("⏳ Loading WES variants from Excel...")
    wes_data = pd.read_excel(input_excel)

    # ------------------------------------------------------------------
    # 2. Load the PrimateAI database
    # ------------------------------------------------------------------
    # The first 11 rows contain metadata rather than variant
    # records, so these rows are skipped
    #
    # `on_bad_lines='skip'` preserves the behaviour of the original
    # workflow by ignoring rows rather than stopping the process
    print("⏳ Loading PrimateAI database (this will take 1-3 minutes)...")
    primate_data = pd.read_csv(
        primate_db_tsv,
        sep='\t',
        skiprows=11,
        low_memory=False,
        on_bad_lines='skip')

    # ------------------------------------------------------------------
    # 3. Check formats before matching
    # ------------------------------------------------------------------
    # Remove an optional 'chr' prefix so that chromosome values use the
    # same representation in both datasets (for example, 'chr10' goes to '10')
    wes_data['Chr'] = (
        wes_data['Chr']
        .astype(str)
        .str.replace('chr', '', case=False)
        .str.strip())

    primate_data['chr'] = (
        primate_data['chr']
        .astype(str)
        .str.replace('chr', '', case=False)
        .str.strip())

    # Convert genomic positions to numeric values so that the merge uses
    # equivalent data types in both datasets
    wes_data['Start'] = pd.to_numeric(wes_data['Start'], errors='coerce')
    primate_data['pos'] = pd.to_numeric(primate_data['pos'], errors='coerce')

    # ------------------------------------------------------------------
    # 4. Match WES variants to PrimateAI predictions
    # ------------------------------------------------------------------
    print("Merging datasets...")

    # A variant is matched using chromosome, genomic position,
    # reference allele and alternate allele
    annotated_data = pd.merge(
        wes_data,
        primate_data,
        left_on=['Chr', 'Start', 'Ref', 'Alt'],
        right_on=['chr', 'pos', 'ref', 'alt'],
        how='left')

    # ------------------------------------------------------------------
    # 5. Keep the original data and add the prediction
    # ------------------------------------------------------------------
    output_columns = wes_data.columns.tolist() + ['primateDL_score']
    annotated_data = annotated_data[output_columns]

    annotated_data.to_csv(output_csv, index=False)

    # Report how many input variants received a PrimateAI score
    matched_variants = annotated_data['primateDL_score'].notna().sum()

    print(f"✅ Success! Annotations saved to {output_csv}")
    print(f"🧬 Variants annotated: {matched_variants} out of {len(wes_data)}")


# Execute the WES PrimateAI pipeline.
perform_primate_annotation(
    input_excel='ExtractedDYS.xlsx',
    primate_db_tsv='PrimateAI_scores_v0.2_hg38.tsv',
    output_csv='DYS__with_PrimateAI.csv')

## 1.2 AlphaMissense

### Method

The AlphaMissense pipeline first identifies the actual header row in the Excel file because the raw workbook may contain rows above the dataset. It then:

1. Extracts and standardises genomic coordinates.
2. Loads the compressed AlphaMissense database directly from the `.gz` file.
3. Matches chromosome and position formats.
4. Matches variants using chromosome, position, reference allele and alternate allele.
5. Adds the AlphaMissense pathogenicity score and class.
6. Saves the annotated data as a CSV file.

In [ ]:
import pandas as pd


def run_alphamissense_pipeline(input_excel, am_db_file, output_csv):
    """Annotate WES variants with AlphaMissense predictions.
    input_excel : str
        Path to the Excel file containing the WES variants.
    am_db_file : str
        Path to the compressed AlphaMissense database.
    output_csv : str
        Path for the annotated CSV output.
    """

    try:
        # ------------------------------------------------------------------
        # 1. Load the Excel file
        # ------------------------------------------------------------------
        # Read without assigning a header because the excel may
        # contain additional rows before the actual column headings
        print("⏳ Loading WES variants from Excel...")
        raw_data = pd.read_excel(input_excel, header=None)

        # Search row-by-row for the first row containing a 'Chr' column.
        header_row_index = None

        for row_index, row in raw_data.iterrows():
            if row.astype(str).str.contains('Chr', case=False).any():
                header_row_index = row_index
                break

        if header_row_index is None:
            print("❌ Error: Could not find a row containing 'Chr' anywhere in the Excel file!")
            return

        print(f"   Found headers at row index {header_row_index}. Re-aligning data...")

        # Use the discovered row as the header and keep all rows below it as
        # the WES variant dataset
        wes_data = raw_data.iloc[header_row_index + 1:].copy()
        wes_data.columns = raw_data.iloc[header_row_index].astype(str).str.strip()

        print(f"   Columns successfully loaded: {wes_data.columns.tolist()}")

        # ------------------------------------------------------------------
        # 2. Load the AlphaMissense database
        # ------------------------------------------------------------------
        # The database is compressed as gzip and is decompressed while being
        # read, so an additional uncompressed copy is not required.
        print("⏳ Loading AlphaMissense Database (this will take 1-3 minutes)...")

        alphamissense_data = pd.read_csv(
            am_db_file,
            sep='\t',
            skiprows=3,
            low_memory=False,
            compression='gzip')

        # Standardise database column names used for genomic matching.
        alphamissense_data.rename(
            columns={'#CHROM': 'chr', 'POS': 'pos', 'REF': 'ref', 'ALT': 'alt'},
            inplace=True)

        # ------------------------------------------------------------------
        # 3. Standardise genomic coordinates
        # ------------------------------------------------------------------
        print("⚙️ Standardising coordinate formatting...")

        wes_data['Chr'] = (
            wes_data['Chr']
            .astype(str)
            .str.replace('chr', '', case=False)
            .str.strip())

        alphamissense_data['chr'] = (
            alphamissense_data['chr']
            .astype(str)
            .str.replace('chr', '', case=False)
            .str.strip())

        wes_data['Start'] = pd.to_numeric(wes_data['Start'], errors='coerce')
        alphamissense_data['pos'] = pd.to_numeric(
            alphamissense_data['pos'],
            errors='coerce')

        # ------------------------------------------------------------------
        # 4. Merge the WES variants with AlphaMissense predictions
        # ------------------------------------------------------------------
        print(
            f"🔗 Merging {len(wes_data)} variants "
            "with AlphaMissense predictions...")

        annotated_data = pd.merge(
            wes_data,
            alphamissense_data,
            left_on=['Chr', 'Start', 'Ref', 'Alt'],
            right_on=['chr', 'pos', 'ref', 'alt'],
            how='left')

        # ------------------------------------------------------------------
        # 5. Select output columns and save the result
        # ------------------------------------------------------------------
        output_columns = wes_data.columns.tolist() + [
            'am_pathogenicity',
            'am_class']
        annotated_data = annotated_data[output_columns]

        annotated_data.to_csv(output_csv, index=False)

        matched_variants = annotated_data['am_pathogenicity'].notna().sum()

        print(f"\n✅ Success! Predictions saved to: {output_csv}")
        print(
            f"🧬 Variants successfully annotated: "
            f"{matched_variants} out of {len(wes_data)}")

    # Error handling
    except FileNotFoundError as error:
        print(
            f"\n❌ Error: Missing file. "
            f"Please check your filenames. Details: {error}"
        )
    except KeyError as error:
        print(
            f"\n❌ Column Error: Could not find expected column "
            f"{error}. Check your Excel layout."
        )
    except Exception as error:
        print(f"\n❌ An unexpected error occurred: {error}")


# Execute the WES AlphaMissense pipeline
run_alphamissense_pipeline(
    input_excel='ExtractedDYS.xlsx',
    am_db_file='AlphaMissense_hg38.tsv.gz',
    output_csv='DYS_AlphaMissense_Annotated.csv'
)

# 2. Twin Study (TWIN)

The same strategy is applied to the twin dataset, but the input format and matching requirements differ slightly from the WES study.

## 2.1 PrimateAI

### Method

For the twin dataset, the PrimateAI pipeline:

1. Reads the TWIN variants from Excel.
2. Extracts chromosome and genomic position from the `chr:pos` field.
3. Extracts reference and alternate from `AAChange.refGene`.
4. Generates the bases.
5. Loads the PrimateAI database.
6. Performs a merge.
7. Filters the merged records.
8. Removes duplicate variant rows.
9. Saves the original TWIN columns plus the PrimateAI score.

In [ ]:
import pandas as pd
import re


def perform_primate_annotation_twin(input_excel, primate_db_tsv, output_csv):
    """Annotate TWIN variants with PrimateAI scores.
    input_excel : str
        Path to the Excel file containing the TWIN variants.
    primate_db_tsv : str
        Path to the PrimateAI tab-separated database.
    output_csv : str
        Path for the annotated CSV output.
    """

    # ------------------------------------------------------------------
    # 1. Load the TWIN variant data
    # ------------------------------------------------------------------
    print("⏳ Loading TWIN variants from Excel...")
    twin_data = pd.read_excel(input_excel)

    # ------------------------------------------------------------------
    # 2. Extract coordinates and alleles
    # ------------------------------------------------------------------
    print("⛏️ Extracting genomic coordinates and alleles...")

    # Split a value such as 'chr14:104949329' into chromosome and position.
    twin_data['Chr'] = (
        twin_data['chr:pos']
        .str.split(':')
        .str[0]
        .str.replace('chr', '', case=False)
        .str.strip())
    twin_data['Start'] = pd.to_numeric(
        twin_data['chr:pos'].str.split(':').str[1],
        errors='coerce')

    def get_alleles(variant_annotation):
        """Extract reference and alternate alleles"""
        if pd.isna(variant_annotation):
            return None, None

        match = re.search(
            r'c\.([ACGT])\d+([ACGT])',
            str(variant_annotation))

        if match:
            return match.group(1), match.group(2)

        return None, None

    # Extract the reference and alternate cDNA bases from the annotation.
    twin_data[['cDNA_Ref', 'cDNA_Alt']] = twin_data.apply(
        lambda row: pd.Series(
            get_alleles(row['AAChange.refGene'])
        ),
        axis=1)

    # Genes may be located on the reverse strand. Generate the DNA
    # complement so that the database can be matched in either orientation
    complement_bases = {
        'A': 'T',
        'T': 'A',
        'C': 'G',
        'G': 'C'}

    twin_data['cDNA_Ref_rev'] = (
        twin_data['cDNA_Ref']
        .map(complement_bases)
        .fillna(twin_data['cDNA_Ref']))
    twin_data['cDNA_Alt_rev'] = (
        twin_data['cDNA_Alt']
        .map(complement_bases)
        .fillna(twin_data['cDNA_Alt']))

    # ------------------------------------------------------------------
    # 3. Load PrimateAI database
    # ------------------------------------------------------------------
    print("⏳ Loading PrimateAI database...")

    primate_data = pd.read_csv(
        primate_db_tsv,
        sep='\t',
        skiprows=11,
        low_memory=False,
        on_bad_lines='skip')

    # Standardise chromosome and position formats before merging.
    primate_data['chr'] = (
        primate_data['chr']
        .astype(str)
        .str.replace('chr', '', case=False)
        .str.strip())
    primate_data['pos'] = pd.to_numeric(
        primate_data['pos'],
        errors='coerce')

    # ------------------------------------------------------------------
    # 4. Merge and filter by DNA strand
    # ------------------------------------------------------------------
    print("🔗 Merging and filtering by DNA strand...")

    # First match by chromosome and position
    merged_data = pd.merge(
        twin_data,
        primate_data,
        left_on=['Chr', 'Start'],
        right_on=['chr', 'pos'],
        how='left')

    # Match either the original allele  or its DNA complement
    allele_match = (
        (
            (merged_data['ref'] == merged_data['cDNA_Ref'])
            & (merged_data['alt'] == merged_data['cDNA_Alt'])
        )
        |
        (
            (merged_data['ref'] == merged_data['cDNA_Ref_rev'])
            & (merged_data['alt'] == merged_data['cDNA_Alt_rev'])
        ))

    # Keep variants where PrimateAI database contains no record,
    # rather than removing those input variants altogether.
    no_database_data = merged_data['ref'].isna()

    # Keep matching records and remove any duplicate input variants.
    annotated_data = (
        merged_data[allele_match | no_database_data]
        .drop_duplicates(subset=['chr:pos'])
        .copy())

    # ------------------------------------------------------------------
    # 5. Select output columns and save the result
    # ------------------------------------------------------------------
    output_columns = twin_data.columns.tolist()[:3] + ['primateDL_score']
    annotated_data = annotated_data[output_columns]

    annotated_data.to_csv(output_csv, index=False)

    matched_variants = annotated_data['primateDL_score'].notna().sum()

    print(f"✅ Success! Annotations saved to {output_csv}")
    print(
        f"🧬 Variants annotated: "
        f"{matched_variants} out of {len(twin_data)}")


# Execute the TWIN PrimateAI pipeline.
perform_primate_annotation_twin(
    input_excel='ExtractedTWIN.xlsx',
    primate_db_tsv='PrimateAI_scores_v0.2_hg38.tsv',
    output_csv='TWIN_with_PrimateAI.csv')

## 2.2 AlphaMissense

### Method

The TWIN AlphaMissense workflow uses the same general database-annotation strategy as the WES workflow, but the input contains a protein-level consequence that can be used directly.

The pipeline:

1. Extracts chromosome and genomic position from `chr:pos`.
2. Extracts the protein substitution from `AAChange.refGene`.
3. Loads the compressed AlphaMissense database.
4. Standardises genomic coordinates.
5. Matches chromosome, position and protein substitution.
6. Removes duplicate variants.
7. Saves the original TWIN columns together with the AlphaMissense score and class.

In [ ]:
import pandas as pd


def run_alphamissense_pipeline_twin(input_excel, am_db_file, output_csv):
    """Annotate TWIN variants with AlphaMissense predictions.
    input_excel : str
        Path to the Excel file containing the TWIN variants.
    am_db_file : str
        Path to the compressed AlphaMissense database.
    output_csv : str
        Path for the annotated CSV output.
    """

    try:
        # ------------------------------------------------------------------
        # 1. Load the TWIN variant data
        # ------------------------------------------------------------------
        print("⏳ Loading TWIN variants from Excel...")
        twin_data = pd.read_excel(input_excel)

        # ------------------------------------------------------------------
        # 2. Extract coordinates and protein variants
        # ------------------------------------------------------------------
        print("⛏️ Extracting genomic coordinates and protein variants...")

        twin_data['Chr'] = (
            twin_data['chr:pos']
            .str.split(':')
            .str[0]
            .str.replace('chr', '', case=False)
            .str.strip())
        twin_data['Start'] = pd.to_numeric(
            twin_data['chr:pos'].str.split(':').str[1],
            errors='coerce')

        # Extract a protein substitution such as 'P1941L' from the full string
        twin_data['protein_variant'] = (
            twin_data['AAChange.refGene']
            .str.extract(r'p\.([A-Z]\d+[A-Z])'))

        # ------------------------------------------------------------------
        # 3. Load the AlphaMissense database
        # ------------------------------------------------------------------
        print(
            "⏳ Loading AlphaMissense Database "
            "(this will take 1-3 minutes to decompress)...")

        alphamissense_data = pd.read_csv(
            am_db_file,
            sep='\t',
            skiprows=3,
            low_memory=False,
            compression='gzip')

        # Standardise the database column names used for matching
        alphamissense_data.rename(
            columns={'#CHROM': 'chr', 'POS': 'pos'},
            inplace=True)

        alphamissense_data['chr'] = (
            alphamissense_data['chr']
            .astype(str)
            .str.replace('chr', '', case=False)
            .str.strip())
        alphamissense_data['pos'] = pd.to_numeric(
            alphamissense_data['pos'],
            errors='coerce')

        # ------------------------------------------------------------------
        # 4. Merge the TWIN variants with AlphaMissense
        # ------------------------------------------------------------------
        print(
            f"🔗 Merging {len(twin_data)} variants "
            "with AlphaMissense predictions...")

        annotated_data = pd.merge(
            twin_data,
            alphamissense_data,
            left_on=['Chr', 'Start', 'protein_variant'],
            right_on=['chr', 'pos', 'protein_variant'],
            how='left')

        # Keep one row per input variant
        annotated_data = annotated_data.drop_duplicates(
            subset=['chr:pos'])

        # ------------------------------------------------------------------
        # 5. Select output columns and save the result
        # ------------------------------------------------------------------
        output_columns = twin_data.columns.tolist()[:3] + [
            'am_pathogenicity',
            'am_class']
        annotated_data = annotated_data[output_columns]

        annotated_data.to_csv(output_csv, index=False)

        matched_variants = annotated_data['am_pathogenicity'].notna().sum()

        print(f"\n✅ Success! Annotations saved to: {output_csv}")
        print(
            f"🧬 Variants successfully annotated: "
            f"{matched_variants} out of {len(twin_data)}")

    except FileNotFoundError as error:
        print(
            f"\n❌ Error: Missing file. "
            f"Please check your filenames. Details: {error}"
        )
    except Exception as error:
        print(f"\n❌ An unexpected error occurred: {error}")


# Execute the TWIN AlphaMissense pipeline
run_alphamissense_pipeline_twin(
    input_excel='ExtractedTWIN.xlsx',
    am_db_file='AlphaMissense_hg38.tsv.gz',
    output_csv='TWIN_AlphaMissense_Annotated.csv')

# 3. EVE

EVE is handled differently because the local database is organised as individual CSV files by gene. The workflow therefore identifies the genes present in the input dataset, loads the corresponding EVE files, and then looks up the relevant protein substitutions.

The final EVE implementation extends the earlier version by constructing an explicit variant-matching field from the EVE database's `wt_aa`, `position` and `mt_aa` columns. It returns both:

- the EVE score (`eve_score_ASM`)
- the 75% retained classification (`eve_class_75`)

The same functions are used for both the DYS and TWIN datasets, with the appropriate input column names for each dataset.

In [ ]:
import pandas as pd
import glob
import os


# Path to the local EVE database.
#
# `os.path.expanduser('~')` keeps the workflow independent of the username
# embedded in a personal macOS file path while preserving the same directory
# structure used by the original analysis
EVE_FOLDER_PATH = os.path.join(
    os.path.expanduser('~'),
    'Documents',
    'University',
    'Dissertation',
    'Code',
    'Final Scripts',
    'EVE_CSVs',
    'EVE_bulk',
    'variant_files')


def load_eve_data_for_genes(unique_genes, eve_folder):
    """Load the EVE CSV corresponding to each gene into a cache."""
    eve_cache = {}

    print(
        f"🔎 Scanning local database for "
        f"{len(unique_genes)} unique genes...")

    for gene in unique_genes:
        # Prefer the exact HUMAN filename used by EVE and fall back to any
        # CSV that begins with the gene symbol
        exact_pattern = os.path.join(eve_folder, f"{gene}_HUMAN.csv")
        fallback_pattern = os.path.join(eve_folder, f"{gene}*.csv")

        matching_files = glob.glob(exact_pattern)

        if not matching_files:
            matching_files = glob.glob(fallback_pattern)

        if matching_files:
            try:
                eve_data = pd.read_csv(
                    matching_files[0],
                    low_memory=False
                )

                # Some EVE files store the protein substitution across three
                # columns. Combine them into a single lookup key, such as
                # 'K776N'
                if all(
                    column in eve_data.columns
                    for column in ['wt_aa', 'position', 'mt_aa']):
                    eve_data['match_variant'] = (
                        eve_data['wt_aa'].astype(str)
                        + eve_data['position'].astype(str)
                        + eve_data['mt_aa'].astype(str))

                eve_cache[gene] = eve_data

            except Exception as error:
                print(
                    f"   ⚠️ Could not read file for {gene}: "
                    f"{error}"
                )
                eve_cache[gene] = None
        else:
            eve_cache[gene] = None

    return eve_cache


def extract_eve_data(gene, variant, eve_cache):
    """Return the EVE score and classification for a variant."""
    if (
        pd.isna(gene)
        or pd.isna(variant)
        or eve_cache.get(gene) is None):
        return pd.Series([None, None])

    eve_data = eve_cache[gene]

    if (
        'match_variant' in eve_data.columns
        and 'EVE_scores_ASM' in eve_data.columns):
        # Match the exact protein substitution
        matching_rows = eve_data[
            eve_data['match_variant'] == variant]

        if not matching_rows.empty:
            score = matching_rows['EVE_scores_ASM'].values[0]

            # Retrieve the 75% retained classification when that field is
            # available in the EVE file
            if 'EVE_classes_75_pct_retained_ASM' in eve_data.columns:
                classification = matching_rows[
                    'EVE_classes_75_pct_retained_ASM'
                ].values[0]
            else:
                classification = None

            return pd.Series([score, classification])

    return pd.Series([None, None])


def run_eve_pipeline_dys(input_excel, eve_folder, output_csv):
    """Annotate the DYS dataset with EVE scores and classes."""
    print("\n--- Starting DYS Pipeline ---")
    print("⏳ Loading DYS variants from Excel...")

    raw_data = pd.read_excel(input_excel, header=None)

    # Locate the header row by searching for 'Chr'
    header_row_index = None

    for row_index, row in raw_data.iterrows():
        if row.astype(str).str.contains('Chr', case=False).any():
            header_row_index = row_index
            break

    if header_row_index is None:
        print("❌ Error: Could not find header row containing 'Chr'.")
        return

    dys_data = raw_data.iloc[header_row_index + 1:].copy()
    dys_data.columns = raw_data.iloc[header_row_index].astype(str).str.strip()

    # Extract the protein substitution required for the EVE lookup
    print("⛏️ Extracting protein variants...")
    dys_data['protein_variant'] = (
        dys_data['AA Change']
        .str.extract(r'p\.([A-Z]\d+[A-Z])'))

    # Load only the EVE files for genes present in the dataset
    unique_genes = dys_data['Gene'].dropna().unique()
    eve_cache = load_eve_data_for_genes(unique_genes, eve_folder)

    # Apply the EVE lookup to each input variant.
    print("📍 Mapping EVE scores to variants...")

    dys_data[['eve_score_ASM', 'eve_class_75']] = dys_data.apply(
        lambda row: extract_eve_data(
            row['Gene'],
            row['protein_variant'],
            eve_cache),
        axis=1)

    # Remove the temporary lookup column and save the table
    final_data = dys_data.drop(columns=['protein_variant'])
    final_data.to_csv(output_csv, index=False)

    matched_variants = final_data['eve_score_ASM'].notna().sum()

    print(f"✅ Success! Annotations saved to {output_csv}")
    print(
        f"🧬 Variants successfully annotated: "
        f"{matched_variants} out of {len(final_data)}")


def run_eve_pipeline_twin(input_excel, eve_folder, output_csv):
    """Annotate the TWIN dataset with EVE scores and classes."""
    print("\n--- Starting TWIN Pipeline ---")
    print("⏳ Loading TWIN variants from Excel...")

    twin_data = pd.read_excel(input_excel)

    # Extract the protein required for the EVE lookup
    print("⛏️ Extracting protein variants...")
    twin_data['protein_variant'] = (
        twin_data['AAChange.refGene']
        .str.extract(r'p\.([A-Z]\d+[A-Z])'))

    # Load only the EVE files for genes present in the TWIN dataset.
    unique_genes = twin_data['Gene.refGene'].dropna().unique()
    eve_cache = load_eve_data_for_genes(unique_genes, eve_folder)

    # Apply the EVE lookup to each input variant.
    print("📍 Mapping EVE scores to variants...")

    twin_data[['eve_score_ASM', 'eve_class_75']] = twin_data.apply(
        lambda row: extract_eve_data(
            row['Gene.refGene'],
            row['protein_variant'],
            eve_cache
        ),
        axis=1)

    final_data = twin_data.drop(columns=['protein_variant'])
    final_data.to_csv(output_csv, index=False)

    matched_variants = final_data['eve_score_ASM'].notna().sum()

    print(f"✅ Success! Annotations saved to {output_csv}")
    print(
        f"🧬 Variants successfully annotated: "
        f"{matched_variants} out of {len(final_data)}")


# Execute the final EVE implementation for both datasets.
run_eve_pipeline_dys(
    input_excel='ExtractedDYS.xlsx',
    eve_folder=EVE_FOLDER_PATH,
    output_csv='DYS_EVE_Annotated.csv')

run_eve_pipeline_twin(
    input_excel='ExtractedTWIN.xlsx',
    eve_folder=EVE_FOLDER_PATH,
    output_csv='TWIN_EVE_Annotated.csv')

# 4. 3D structural analysis

This section extends the variant annotation into 3D protein structure analysis.

The pipeline takes a targeted variant table and, for each gene:

1. identifies the reviewed human UniProt accession;
2. translates the genomic variant coordinate into an amino-acid position using Ensembl Variant Effect Predictor (VEP);
3. downloads the corresponding AlphaFold predicted protein structure in PDB format and
4. generates a PyMOL script that highlights the variant residue(s) on the protein structure.

The structural workflow is intended to support visual inspection of where variants occur within a protein. The generated PyMOL scripts do not perform a quantitative structural analysis; instead, they provide a reproducible way to prepare the structures for visualisation.

> **External resources used:** UniProt REST API, Ensembl REST API/VEP and the AlphaFold Protein Structure Database.

> **Input required:** `Targeted_Variants.csv`, containing the columns `Gene`, `Chr`, `Start` and `Alt`.

> **Output folders:** PDB structures are saved to `pdb_files/` and PyMOL scripts are saved to `pymol_scripts/`.

## 4.1 Imports and output directories

The structural workflow uses `requests` to communicate with the external UniProt, Ensembl and AlphaFold services. A short delay is retained between API requests to reduce the frequency of requests made to external services.

In [ ]:
import os
import time

import pandas as pd
import requests


# Create local directories for downloaded protein structures and PyMOL scripts
os.makedirs("pdb_files", exist_ok=True)
os.makedirs("pymol_scripts", exist_ok=True)

## 4.2 Retrieve the reviewed UniProt

In [ ]:
def get_uniprot_accession(gene_symbol):

    url = (
        "https://rest.uniprot.org/uniprotkb/search?"
        f"query=(gene:{gene_symbol})+AND+(reviewed:true)+AND+(organism_id:9606)"
        "&format=json")

    response = requests.get(url)

    if response.status_code == 200:
        results = response.json().get("results", [])

        if results:
            return results[0]["primaryAccession"]

    return None

## 4.3 Translate genomic coordinates to amino-acid positions

PyMOL requires residue positions within the protein sequence, whereas the targeted dataset contains genomic coordinates.

Ensembl VEP is therefore queried for each variant. The first transcript consequence containing a `protein_start` value is used as the amino-acid position for the structural visualisation.

In [ ]:
def get_amino_acid_position(chromosome, genomic_position, alternate_allele):
    """Translate a genomic variant coordinate into a protein residue position.
    chromosome : str
        Chromosome identifier without the ``chr`` prefix.
    genomic_position : int or str
        Genomic position of the variant.
    alternate_allele : str
        Alternate allele.

    Returns
    -------
    int or None
        Protein residue position from the first applicable VEP transcript
        consequence, or ``None`` when no protein position is returned.
    """
    server = "https://rest.ensembl.org"
    endpoint = (
        f"/vep/human/region/"
        f"{chromosome}:{genomic_position}-{genomic_position}/"
        f"{alternate_allele}?")
    headers = {"Content-Type": "application/json"}

    response = requests.get(server + endpoint, headers=headers)

    if response.status_code == 200:
        response_data = response.json()

        if response_data and "transcript_consequences" in response_data[0]:
            for transcript_consequence in response_data[0]["transcript_consequences"]:
                if "protein_start" in transcript_consequence:
                    return transcript_consequence["protein_start"]

    return None

## 4.4 Download the AlphaFold structure

Once a UniProt accession has been identified, the corresponding AlphaFold predicted structure is downloaded in PDB format.

Existing PDB files are reused rather than downloaded again. This makes repeated runs more efficient and avoids unnecessary requests.

In [ ]:
def download_alphafold_pdb(uniprot_accession):
    """Download the AlphaFold PDB structure for a UniProt accession.
    uniprot_accession : str
        Primary UniProt accession.

    Returns
    -------
    str or None
        Local PDB filepath when available; otherwise ``None``.
    """
    url = (
        "https://alphafold.ebi.ac.uk/files/"
        f"AF-{uniprot_accession}-F1-model_v4.pdb")
    pdb_filepath = f"pdb_files/{uniprot_accession}.pdb"

    # Reuse a previously downloaded structure where possible
    if not os.path.exists(pdb_filepath):
        response = requests.get(url)

        if response.status_code == 200:
            with open(pdb_filepath, "wb") as pdb_file:
                pdb_file.write(response.content)

            return pdb_filepath

    return pdb_filepath if os.path.exists(pdb_filepath) else None

## 4.5 Generate a PyMOL visualisation script

For each gene with mapped amino-acid positions, a `.pml` file is created for use in PyMOL.

The generated script:

- loads the AlphaFold structure;
- hides the default representation;
- displays the protein as a cartoon;
- colours the structure grey;
- selects the variant residue positions;
- displays those residues as spheres;
- colours the variant residues red;
- labels the alpha carbon with residue name and position; and
- prepares a white-background, zoomed and ray-traced view.

The script is deliberately generated as a separate file so that structural visualisation remains reproducible and can be rerun directly in PyMOL.

In [ ]:
def generate_pymol_script(gene_symbol, uniprot_accession, amino_acid_positions):
    """Create a PyMOL script that highlights candidate variant residues.
    gene_symbol : str
        Gene symbol used to name the PyMOL object and output script.
    uniprot_accession : str
        UniProt accession corresponding to the downloaded PDB structure.
    amino_acid_positions : list
        Protein residue positions to highlight.

    Returns
    -------
    str
        Path to the generated PyMOL script.
    """
    pymol_script_path = f"pymol_scripts/{gene_symbol}_visualization.pml"

    # PyMOL accepts multiple residue identifiers separated by '+'
    positions_string = "+".join(
        str(position)
        for position in amino_acid_positions
        if position)

    with open(pymol_script_path, "w") as pymol_file:
        pymol_file.write(
            f"load ../pdb_files/{uniprot_accession}.pdb, {gene_symbol}\n")
        pymol_file.write("hide everything, all\n")
        pymol_file.write(f"show cartoon, {gene_symbol}\n")
        pymol_file.write("color gray80, all\n")

        if positions_string:
            pymol_file.write(
                f"select variants, {gene_symbol} and resi {positions_string}\n")
            pymol_file.write("show spheres, variants\n")
            pymol_file.write("color red, variants\n")

            pymol_file.write("label variants and name CA, resn+resi\n")
            pymol_file.write("set label_color, black\n")
            pymol_file.write("set label_position, (2, 2, 2)\n")

        pymol_file.write("bg_color white\n")
        pymol_file.write("zoom variants\n")
        pymol_file.write("ray 1200, 1200\n")

    return pymol_script_path

## 4.6 Run the structural pipeline

The input table is processed row by row. UniProt accessions are cached at the gene level, while amino-acid positions are collected for each gene.

A one-second pause is retained after each external request, matching the original workflow and helping to avoid sending repeated requests to the public APIs too quickly.

Only genes with both a UniProt accession and at least one successfully mapped amino-acid position proceed to structure download and PyMOL script generation.

In [ ]:
print("🟢 Starting targeted structural pipeline...")

# Load the targeted variant dataset
targeted_variants = pd.read_csv("Targeted_Variants.csv")

# Store the UniProt accession and mapped protein positions for each gene
gene_structure_data = {}

# Process each variant and map it from genomic to protein coordinates
for _, variant in targeted_variants.iterrows():
    gene_symbol = variant["Gene"]
    chromosome = str(variant["Chr"]).replace("chr", "")
    genomic_position = variant["Start"]
    alternate_allele = variant["Alt"]

    # Retrieve the UniProt accession once per gene
    if gene_symbol not in gene_structure_data:
        print(f"📥 Fetching UniProt ID for {gene_symbol}...")
        uniprot_accession = get_uniprot_accession(gene_symbol)

        gene_structure_data[gene_symbol] = {
            "uniprot": uniprot_accession,
            "amino_acid_positions": [],}

        # Set a short delay to avoid overwhelming the UniProt API with rapid requests
        time.sleep(1)

    if gene_structure_data[gene_symbol]["uniprot"]:
        print(
            f"🔁 Translating coordinate "
            f"{chromosome}:{genomic_position} for {gene_symbol}...")

        amino_acid_position = get_amino_acid_position(
            chromosome,
            genomic_position,
            alternate_allele,)

        if amino_acid_position:
            gene_structure_data[gene_symbol]["amino_acid_positions"].append(
                amino_acid_position)

        time.sleep(1)

# Download structures and create a PyMOL script for each eligible gene
for gene_symbol, structure_data in gene_structure_data.items():
    uniprot_accession = structure_data["uniprot"]
    amino_acid_positions = structure_data["amino_acid_positions"]

    if uniprot_accession and amino_acid_positions:
        print(
            f"⬇️ Downloading AlphaFold structure for "
            f"{gene_symbol} ({uniprot_accession})...")
        download_alphafold_pdb(uniprot_accession)

        print(f"🪄 Generating PyMOL script for {gene_symbol}...")
        generate_pymol_script(
            gene_symbol,
            uniprot_accession,
            amino_acid_positions,)

print(
    "🏁 Pipeline complete. Open PyMOL and run the generated .pml scripts "
    "from the pymol_scripts folder.")

## 4.7 Expected structural outputs

After a successful run, the structural workflow produces:

- `pdb_files/<UniProt accession>.pdb` — the downloaded AlphaFold structure;
- `pymol_scripts/<gene>_visualization.pml` — the PyMOL commands used to highlight the mapped variant residues.

The PDB files can be opened directly in PyMOL or another structural biology viewer. The `.pml` files can be run from PyMOL to reproduce the visualisation.

# 5. External resources and downloads

The  workflows use the following external resources. The links below point to the official source pages or official download locations identified for the resources used by this notebook.

### PrimateAI

**File used by this notebook:** `PrimateAI_scores_v0.2_hg38.tsv` (the HG38/GRCh38 exome-wide score file).


- [PrimateAI project and download information](https://github.com/Illumina/PrimateAI)
- [Illumina BaseSpace download](https://basespace.illumina.com/s/yYGFdGih1rXL)


### AlphaMissense

**File used by this notebook:** `AlphaMissense_hg38.tsv.gz`.


- [Official AlphaMissense repository and prediction data](https://github.com/google-deepmind/alphamissense)
- [AlphaMissense HG38 score file](https://storage.googleapis.com/dm_alphamissense/AlphaMissense_hg38.tsv.gz)


### EVE

**Data used by this notebook:** EVE bulk data extracted into gene/protein-specific CSV files.

The EVE website provides a bulk download containing variant data and related resources. The notebook reads the downloaded gene-specific CSV files locally rather than querying EVE online.

- [EVE bulk data download](https://evemodel.org/download/bulk)
- [EVE bulk download endpoint](https://evemodel.org/api/proteins/bulk/download/)

### AlphaFold Protein Structure Database


- [AlphaFold Protein Structure Database](https://alphafold.ebi.ac.uk/)
- [AlphaFold DB FAQ and download information](https://alphafold.ebi.ac.uk/faq)

### UniProt and Ensembl


- [UniProt REST API](https://rest.uniprot.org/)
- [Ensembl Variant Effect Predictor (VEP) REST service](https://rest.ensembl.org/)


# 6. Outputs

After successful execution, the notebook produces the following annotated CSV files:

- `DYS__with_PrimateAI.csv`
- `DYS_AlphaMissense_Annotated.csv`
- `TWIN_with_PrimateAI.csv`
- `TWIN_AlphaMissense_Annotated.csv`
- `DYS_EVE_Annotated.csv`
- `TWIN_EVE_Annotated.csv`

The structural workflow additionally produces:

- `pdb_files/<UniProt accession>.pdb` — AlphaFold coordinate files downloaded during execution.
- `pymol_scripts/<gene>_visualization.pml` — PyMOL scripts generated to reproduce the structural visualisation.

Each annotation pipeline reports the number of variants for which an annotation was successfully retrieved. These messages provide a simple check that the expected matching process has occurred.